1- Her csv'yi tek tek tara: Baseline:602 satır ve 140 sütun, Attack:602 satır ve 141 sütun. Uymayan veya missing value içerenleri listele.
2- Metadata'yı filtrele: Sorunlu csv'leri metadata'dan drop et --> clean_metadata.csv kaydet
3- Class imbalance kontrolü - Dosya bazında: clean_metadata üzerinden "Scenario" dağılımı (EDA'da yaptığın gibi).
4- Class imbalance kontrolü - Satır bazında: Her csv'ye girerek result_label.value_counts() topla --> gerçek label dağılımı.
5- Her temiz csv'yi işle & kaydet: Her csv'yi aç --> selected features + result_label kolonlarını seç --> label_binary ekle(baseline-> 0 , diğerleri -> 1) --> ML training'te kullanmak üzere kaydet.

In [33]:
import pandas as pd
from pathlib import Path

In [34]:
BASE = Path("../data/raw")

meta_data_df = pd.read_csv(BASE / "all_metadata.csv")

EXPECTED_ROWS = 602
EXPECTED_COLS_BASELINE = 140 
EXPECTED_COLS_ATTACK = 141


# 1- Her csv'yi tek tek tara, Uymayan veya missing value içerenleri listele.

In [35]:
# önce metadataki tüm baseline ve attack csv dosyalarını birer birer gezeceğiz ve satır-sütun uyumsuzluğu yapanları veya null değer içerenleri listeleceğiz.abs

#Sorunlu csv'leri tutacağımız liste:
problematic_csvs = []

for _, row in meta_data_df.iterrows():
    #csv'yi oku:
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path)

    if row["Scenario"] == "baseline":
        expected_cols = EXPECTED_COLS_BASELINE
    else:
        expected_cols = EXPECTED_COLS_ATTACK

    #Kontroller
    row_ok = df.shape[0] == EXPECTED_ROWS 
    col_ok = df.shape[1] == expected_cols
    #missing değer var mı  kontrolü:
    missing_count = df.isnull().sum().sum() # ilk sum sütun sütun kaç hücrenin null değere sahip olduğunu hesağlarken ikinci sum totali hesaplıyo

    #Sorun varsa listeye ekle:
    if not row_ok or not col_ok or missing_count > 0:
        problematic_csvs.append({
            "data_file"  : row["data_file"],
            "Scenario"   : row["Scenario"],
            "actual_rows": df.shape[0],
            "actual_cols": df.shape[1],
            "expected_rows" : EXPECTED_ROWS,
            "expected_cols" : expected_cols,
            "missing_values" : missing_count

        })


In [36]:
problematic_df = pd.DataFrame(problematic_csvs)

print(f"Toplam taranan csv : {len(meta_data_df)}")
print(f"Sorunlu csv sayısı: {len(problematic_df)}")

# pd.set_option("display.max_rows", None) --> 194 uyumsuz csv'nin tamamını incelemek için yorum satırından çıkarabilirsiniz.
display(problematic_df)

Toplam taranan csv : 1920
Sorunlu csv sayısı: 194


,data_file,Scenario,actual_rows,actual_cols,expected_rows,expected_cols,missing_values
0,./data/raw/baseline/fsw_data_4_180.0_0.csv,baseline,602,140,602,140,4214
1,./data/raw/baseline/fsw_data_5_72.0_0.csv,baseline,602,140,602,140,4214
2,./data/raw/baseline/fsw_data_5_180.0_0.csv,baseline,602,140,602,140,4214
3,./data/raw/baseline/fsw_data_5_252.0_0.csv,baseline,602,140,602,140,4214
4,./data/raw/baseline/fsw_data_6_36.0_0.csv,baseline,602,140,602,140,4214
...,...,...,...,...,...,...,...
189,./data/raw/attack_rwc/fsw_data_9_288.0_-50.csv,attack_rwc,602,141,602,141,4214
190,./data/raw/attack_rwc/fsw_data_9_288.0_-25.csv,attack_rwc,602,141,602,141,4214
191,./data/raw/attack_rwc/fsw_data_9_288.0_0.csv,attack_rwc,602,141,602,141,4214
192,./data/raw/attack_rwc/fsw_data_9_288.0_25.csv,attack_rwc,602,141,602,141,4214


In [37]:
# metadata EDA'sında FSW_rows
#602    1918
#510       1
#341       1 şöyle bir çıktı almıştık.
# bu çıktıyı doğrular şekilde 188 ve 132. satırda bu 2 eksik satırlı csv dosyası gözüküyo! 

# ayrıca metadata EDA'sında columnlar da 192 dosyada eksik çıkmıştı 7 tane, burada da aynı şeyi görüyoruz 4214/612 = 7 yapıyor her satırda 7 column eksik yani bakalım hangi kolonlarmış onlar:
#örnek bir missing value içeren csv dosyasını; hangi kolonları eksikmiş diye bakmak için inceliyoruz: 
df = pd.read_csv("../data/raw/baseline/fsw_data_4_180.0_0.csv")

# Hangi kolonlarda NaN var?
bos_kolonlar = df.columns[df.isnull().any()].tolist()
print(bos_kolonlar)

print(df[bos_kolonlar].isnull().sum())


['RawPrimaryGenericPointData[0]()', 'RawPrimaryGenericPointData[1]()', 'RawPrimaryGenericPointData[2]()', 'RawSecondaryGenericPointData[0]()', 'RawSecondaryGenericPointData[1]()', 'RawSecondaryGenericPointData[2]()', 'AngleToPrimaryTarget(rad)']
RawPrimaryGenericPointData[0]()      602
RawPrimaryGenericPointData[1]()      602
RawPrimaryGenericPointData[2]()      602
RawSecondaryGenericPointData[0]()    602
RawSecondaryGenericPointData[1]()    602
RawSecondaryGenericPointData[2]()    602
AngleToPrimaryTarget(rad)            602
dtype: int64


In [38]:
# Önceki hücrede missing value gösteren kolonların neler olduğunu inceledik, bu kolonlar bizim halihazırda seçtiğimiz featurelara ait olmadığı için,
#tüm metadata'yı bu sefer satır,sütun ve seçtiğimiz featurelara ait kolonların eksik olup olmadığına göre tekrar kontrol ettik. yani o missing values
# değeri bizim için bir şey ifade etmiyor halihazırda seçmediğimiz featurelara ait oldukları için.

SELECTED_FEATURES = [
    "Q_B_I[0](1)", "Q_B_I[1](1)", "Q_B_I[2](1)", "Q_B_I[3](1)", 
"AttitudeError[0](rad)", "AttitudeError[1](rad)", "AttitudeError[2](rad)",
"AngVel_B_I[0](rad/sec)", "AngVel_B_I[1](rad/sec)", "AngVel_B_I[2](rad/sec)", "AngVelMag_B_I(rad/sec)",
"SensedWheelSpeed__RWA_A(rad/sec)", "SensedWheelSpeed__RWA_B(rad/sec)", "SensedWheelSpeed__RWA_C(rad/sec)",
"WheelCmd__RWA_A(N*m)", "WheelCmd__RWA_B(N*m)", "WheelCmd__RWA_C(N*m)",
"DesiredWheelCommand[0](N*m)", "DesiredWheelCommand[1](N*m)", "DesiredWheelCommand[2](N*m)",
"TotalTorqueRodCommand[0](A*m^2)", "TotalTorqueRodCommand[1](A*m^2)", "TotalTorqueRodCommand[2](A*m^2)",
"BField_B__TAM[0](T)", "BField_B__TAM[1](T)", "BField_B__TAM[2](T)"
]

#bu sefer seçtiğimiz feature'lar eksik mi diye tekrar tarıyoruz, değilse column sayısı genele göre eksik olsa da o csv dosyasını elemeyeceğiz çünkü bizim featurelarımızı içeriyor olacak.
problematic_csvs_final = []

for _, row in meta_data_df.iterrows():
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path)

    if row["Scenario"] == "baseline":
        expected_cols = EXPECTED_COLS_BASELINE
    else:
        expected_cols = EXPECTED_COLS_ATTACK


    row_ok = df.shape[0] == EXPECTED_ROWS
    col_ok = df.shape[1] == expected_cols

    # Sadece seçilen feature'lardaki missing value'lara bakıyoruz
    missing_in_features = df[SELECTED_FEATURES].isnull().sum().sum()

    if not row_ok or not col_ok or missing_in_features > 0:
        problematic_csvs_final.append({
            "data_file"           : row["data_file"],
            "Scenario"            : row["Scenario"],
            "actual_rows"         : df.shape[0],
            "expected_rows"       : EXPECTED_ROWS,
            "actual_cols"         : df.shape[1],
            "expected_cols"       : expected_cols,
            "missing_in_features" : missing_in_features
        })

In [39]:
problematic_df_final = pd.DataFrame(problematic_csvs_final)

print(f"Toplam taranan csv : {len(meta_data_df)}")
print(f"Sorunlu csv sayısı: {len(problematic_df_final)}")

display(problematic_df_final)

Toplam taranan csv : 1920
Sorunlu csv sayısı: 2


,data_file,Scenario,actual_rows,expected_rows,actual_cols,expected_cols,missing_in_features
0,./data/raw/attack_rwb/fsw_data_12_144.0_-50.csv,attack_rwb,510,602,141,141,1530
1,./data/raw/attack_rwc/fsw_data_9_108.0_-50.csv,attack_rwc,341,602,141,141,1023


In [40]:
# Bizim seçtiğimiz featurelara dair missing value barındıran sadece 2 csv dosyası çıktı, bunları eleyip clean metadata oluşturacağım.


# 2- Metadata'yı filtrele: Sorunlu csv'leri metadata'dan drop et

In [41]:
files_to_drop = problematic_df_final["data_file"].tolist()

# Metadata'daki her satır için: bu satırın data_file'ı drop listesinde mi?
is_problematic = meta_data_df["data_file"].isin(files_to_drop)
clean_metadata_df = meta_data_df[~is_problematic]

print(f"Original metadata: {len(meta_data_df)} satır")
print(f"Cleaned metadata: {len(clean_metadata_df)} satır")
print(f"Drop Edilen: {len(meta_data_df) - len(clean_metadata_df)} satır")

Original metadata: 1920 satır
Cleaned metadata: 1918 satır
Drop Edilen: 2 satır


In [42]:
#clean_metadata'yı ayrı bir yere kaydet:
import os
os.makedirs("../data/processed", exist_ok = True)

clean_metadata_df.to_csv("../data/processed/clean_metadata.csv", index = False)

# 3- Drop'tan sonra Class Balance

In [43]:
print("Scenario-Wise Class Distribution (File Level):")
print(clean_metadata_df["Scenario"].value_counts())

Scenario-Wise Class Distribution (File Level):
Scenario
attack_rwa    600
attack_rwb    599
attack_rwc    599
baseline      120
Name: count, dtype: int64


In [47]:
all_dfs = []

for _, row in clean_metadata_df.iterrows():
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path, usecols=["result_label"])
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index = True)

print("Scenario-Wise Class Distribution (Row Level):")
print(combined_df["result_label"].value_counts())

Scenario-Wise Class Distribution (Row Level):
result_label
baseline    615236
RWA         180000
RWB         179700
RWC         179700
Name: count, dtype: int64


# 5- Her temiz csv'yi işle & kaydet:

In [50]:
processed_paths = []

for _, row in clean_metadata_df.iterrows():
    # 1- CSV'yi aç
    file_path = Path("..") / row["data_file"].lstrip("./")
    df = pd.read_csv(file_path)

    # 2- Sadece seçilen 26 feature + result_label kolonlarını al
    df_processed = df[SELECTED_FEATURES + ["result_label"]]

    # 3- label_binary ekle: baseline → 0, attack → 1
    df_processed = df_processed.copy() # SettingWithCopyWarning'i önlemek için
    df_processed["label_binary"] = (df_processed["result_label"] != "baseline").astype(int)

    # 4- Kaydet: orijinal dosya adını ve senaryo klasör yapısını koru
    file_name = Path(row["data_file"]).name #Uzun dosya yolunun sadece en sonundaki dosya adını çeker. Örneğin: fsw_data_1_0.0_0.csv 
    scenario = row["Scenario"]
    save_path = Path("../data/processed") / scenario / file_name

    df_processed.to_csv(save_path, index= False)
    processed_paths.append(str(save_path))

print(f"{len(processed_paths)} csv dosyası işlendi ve kaydedildi.")


1918 csv dosyası işlendi ve kaydedildi.


In [ ]:
# data_file_processed kolonunu clean_metadata_df'e ekle
# (SettingWithCopyWarning'i önlemek için önce copy alıyoruz)
clean_metadata_df = clean_metadata_df.copy()
clean_metadata_df["data_file_processed"] = processed_paths

# clean_metadata.csv'yi data_file_processed kolonuyla birlikte güncelle
clean_metadata_df.to_csv("../data/processed/clean_metadata.csv", index=False)